# Relative Longitude World Overlay

Visual check for the engineered location proxies. This notebook reads `data/place_relative_longitude.csv` and plots the relative longitude / latitude proxy points on a world map.

**Important:** the longitude and latitude squishing below is only for visualization. It does not modify `clean_train.csv`, `clean_val.csv`, `clean_test.csv`, or the modeling features.


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

RELATIVE_LONGITUDE_PATH = Path("data/place_relative_longitude.csv")

# Display-only tuning. These values make the point cloud easier to inspect on a world map.
# They are intentionally not written back to any cleaned modeling dataset.
LONGITUDE_SCALE_DEG_PER_UNIT = 6.0
LONGITUDE_SHIFT_DEG = 20.0
LONGITUDE_SIGN = 1
LONGITUDE_VISUAL_SQUISH = 0.20  # longitude is squished by 20% for the map only
LATITUDE_VISUAL_SQUISH = 0.05   # latitude is squished by 5% for the map only
MIN_COMPONENT_SIZE = 1


In [ ]:
places = pd.read_csv(RELATIVE_LONGITUDE_PATH)

required_cols = ["Place_ID", "latitude", "relative_longitude", "latitude_band"]
missing_cols = [col for col in required_cols if col not in places.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

places.head()


In [ ]:
def wrap_longitude(lon):
    return ((lon + 180) % 360) - 180


def build_overlay_df(
    longitude_scale=LONGITUDE_SCALE_DEG_PER_UNIT,
    longitude_shift=LONGITUDE_SHIFT_DEG,
    longitude_sign=LONGITUDE_SIGN,
    longitude_visual_squish=LONGITUDE_VISUAL_SQUISH,
    latitude_visual_squish=LATITUDE_VISUAL_SQUISH,
    min_component_size=MIN_COMPONENT_SIZE,
):
    overlay = places.copy()
    overlay = overlay[overlay["longitude_component_size"] >= min_component_size].copy()

    # Visualization-only coordinates. These columns are not modeling features.
    overlay["map_longitude"] = wrap_longitude(
        longitude_shift
        + longitude_sign
        * overlay["relative_longitude"]
        * longitude_scale
        * (1 - longitude_visual_squish)
    )
    overlay["map_latitude"] = overlay["latitude"] * (1 - latitude_visual_squish)
    return overlay


overlay_df = build_overlay_df()
overlay_df[["Place_ID", "latitude", "relative_longitude", "map_latitude", "map_longitude"]].head()


In [ ]:
fig = px.scatter_geo(
    overlay_df,
    lat="map_latitude",
    lon="map_longitude",
    color="latitude_band",
    hover_name="Place_ID",
    hover_data={
        "latitude": ":.2f",
        "relative_longitude": ":.2f",
        "map_latitude": ":.2f",
        "map_longitude": ":.2f",
        "longitude_component_size": True,
    },
    projection="natural earth",
    title="Visualization-only World Overlay of Engineered Location Proxies",
)
fig.update_traces(marker={"size": 6, "opacity": 0.75})
fig.update_layout(legend_title_text="Latitude band")
fig.show()


## Reminder

The map uses `map_longitude` and `map_latitude`, which are display-only columns. The model still uses the engineered proxy features from the cleaned data, without this 20% longitude squish or 5% latitude squish.
